# BTP II 

STEP 1 — GLOBAL TEMPORAL WINDOWING

In [1]:
import pandas as pd 
import numpy as np

In [2]:
df = pd.read_excel(r"C:\Users\Dell\Documents\Elevator BTP\elevator_fault_dataset_full.xlsx")
df.head()

,Timestamp,Motor_Current,Motor_Voltage,Vibration_Level,Speed,Cabin_Position,Cabin_Acceleration,Cabin_Deceleration,Door_Operation_Time,Braking_Force,...,Control_System_Delay,Frequency_of_Emergency_Stops,Button_Press_Logs,Door_Obstruction_Events,Overload_Events,Health_Indices,RUL,Fault_State,Fault_Category,Fault_Severity
0,2020-01-01 00:00:00,22.289903,241.253222,1.393233,3.519079,19,0.349191,0.358643,5.747832,112.280046,...,0.097096,3,61.037683,0,0,171.975594,893.941155,1,Door Failure,Minor
1,2020-01-01 01:00:00,25.240487,236.078046,0.090303,2.226921,9,0.260127,0.266565,7.552277,142.770141,...,0.075588,1,15.653676,0,0,150.250886,2130.455491,0,Brake Failure,Critical
2,2020-01-01 02:00:00,32.769471,432.220975,0.932987,2.523211,20,0.291057,0.161099,7.703547,98.016876,...,0.091237,3,25.963784,0,0,164.327588,2472.445475,0,Motor Malfunction,Minor
3,2020-01-01 03:00:00,34.913962,645.108627,0.490413,4.910428,20,0.439094,0.214058,8.335525,75.588623,...,0.128085,2,25.590539,0,0,142.256458,1753.632090,0,Motor Malfunction,Moderate
4,2020-01-01 04:00:00,27.799073,224.451891,0.844176,5.188226,20,0.566116,0.365294,4.717031,95.882924,...,0.112181,2,41.910883,0,0,89.765157,2399.361778,0,Sensor Error,Critical


In [3]:
df['Age_Difference'] = df['Component_Age'].diff()
is_new_component = (df['Age_Difference'] < 0) | (df.index == df.index.min())
df['Component_ID_Proxy'] = is_new_component.cumsum()
df = df.drop(columns=['Age_Difference'])
print("Component_ID_Proxy successfully created.")

Component_ID_Proxy successfully created.


In [4]:
df = df.sort_values("Timestamp").reset_index(drop=True)

In [7]:
feature_cols = [
    col for col in df.columns
    if col not in ["Elevator_ID", "Timestamp", "Fault_State", "RUL"]
]

In [8]:
def create_global_windows(df, feature_cols, window_size=20):
    X, y_fault, y_rul = [], [], []

    data = df[feature_cols].values
    fault_labels = df["Fault_State"].values
    rul_values = df["RUL"].values

    for i in range(len(df) - window_size):
        X.append(data[i:i+window_size])
        y_fault.append(fault_labels[i+window_size])
        y_rul.append(rul_values[i+window_size])

    return np.array(X), np.array(y_fault), np.array(y_rul)

In [9]:
X, y_fault, y_rul = create_global_windows(df, feature_cols, 20)

In [10]:
split_idx = int(0.7 * len(X))

X_train, X_test = X[:split_idx], X[split_idx:]
y_fault_train, y_fault_test = y_fault[:split_idx], y_fault[split_idx:]
y_rul_train, y_rul_test = y_rul[:split_idx], y_rul[split_idx:]

In [11]:
print(X_train.shape, X_test.shape)
print(np.mean(y_rul_train), np.mean(y_rul_test))

(30142, 20, 44) (12919, 20, 44)
1991.897854315232 1996.562071862444


Fixing mean (test) > mean (train)

In [12]:
rul_threshold = np.percentile(y_rul, 40)  # bottom 40% = degraded
print(rul_threshold)

1777.65221962648


In [13]:
train_idx = y_rul > rul_threshold
test_idx  = y_rul <= rul_threshold

X_train, X_test = X[train_idx], X[test_idx]
y_rul_train, y_rul_test = y_rul[train_idx], y_rul[test_idx]
y_fault_train, y_fault_test = y_fault[train_idx], y_fault[test_idx]

In [14]:
print(np.mean(y_rul_train), np.mean(y_rul_test))

2495.0680369206393 1240.6846286954433
